# Ch 12 — 아이리스 품종 예측 (다중분류)

원본: `03_Iris_Multi_Classfication.py`

다루는 내용:
1. CSV 로드 + 컬럼 이름
2. seaborn pairplot으로 클래스 분리도 시각화
3. LabelEncoder + one-hot
4. Dense(16) → Dense(3, softmax)
5. categorical_crossentropy 손실

## 0. 환경

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

import seaborn as sns
from sklearn.preprocessing import LabelEncoder

## 1. 데이터

In [ ]:
DATA = "../../data/iris.csv"
columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
df = pd.read_csv(DATA, names=columns)
print("shape:", df.shape)
print("classes:", df["species"].unique())
df.head()

## 2. pairplot — 클래스 분리도

In [ ]:
sns.pairplot(df, hue="species", height=1.8)
plt.show()

**관찰:** `setosa` 는 `petal_length`/`petal_width` 만으로도 거의 완벽하게 분리됨.
나머지 두 종은 일부 겹침이 있어 모델의 학습 대상은 사실상 그 두 종 구분.

## 3. 라벨 인코딩 + 원-핫

In [ ]:
X = df.iloc[:, 0:4].to_numpy(dtype="float32")
y_str = df["species"].to_numpy()

encoder = LabelEncoder()
y_int = encoder.fit_transform(y_str)
print("classes (LabelEncoder):", list(encoder.classes_))
print("y_int 처음 5개:", y_int[:5])

y_onehot = keras.utils.to_categorical(y_int, num_classes=3)
print("y_onehot shape:", y_onehot.shape)
print("y_onehot 처음 3행:\n", y_onehot[:3])

## 4. 모델 — softmax 출력

In [ ]:
keras.utils.set_random_seed(0)
model = Sequential([
    Input(shape=(4,)),
    Dense(16, activation="relu"),
    Dense(3, activation="softmax"),
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 5. 학습

In [ ]:
hist = model.fit(X, y_onehot, epochs=50, batch_size=1, verbose=0)
print(f"final train accuracy: {hist.history['accuracy'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1); plt.plot(hist.history["loss"]); plt.title("loss"); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2); plt.plot(hist.history["accuracy"]); plt.title("accuracy"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. 예측 디코딩

In [ ]:
probs = model.predict(X[:5], verbose=0)
preds = np.argmax(probs, axis=1)
print("확률 (5개 × 3 클래스):\n", probs.round(3))
print("예측 정수:", preds)
print("예측 라벨:", encoder.inverse_transform(preds))
print("정답 라벨:", y_str[:5])

## 마무리
- [ ] sparse_categorical_crossentropy 로 바꾸면 원-핫 없이도 학습 가능 (라벨이 정수면 됨)
- [ ] hidden layer를 빼면 (Linear → Softmax) 정확도가 어떻게 변하는지
- [ ] `batch_size=1` 의 효과 — SGD에 가까워서 학습이 매우 느리지만 변동이 큼